In [0]:
bronze_df = spark.table("workspace.default.bronze_retail_sales")

display(bronze_df)

In [0]:
from pyspark.sql.functions import col, count, when

bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_df.columns
]).show()

In [0]:
silver_df = bronze_df.dropDuplicates(["Order_ID"])

print("Records after removing duplicates:", silver_df.count())

Records after removing duplicates: 9


In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.filter(
    col("Customer_ID").isNotNull()
)

silver_df = silver_df.filter(
    col("Quantity").isNotNull()
)

silver_df = silver_df.fillna({
    "City": "Unknown"
})

print("Records after cleaning:", silver_df.count())

display(silver_df)

Records after cleaning: 7


Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore
1009,2026-01-13,C108,Mouse,Accessories,2,800,Unknown


In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "Revenue",
    col("Quantity") * col("Price")
)

display(silver_df)

Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City,Revenue
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune,55000
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai,50000
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune,4500
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi,1600
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai,12000
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore,55000
1009,2026-01-13,C108,Mouse,Accessories,2,800,Unknown,1600


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_retail_sales")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_retail_sales;

Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City,Revenue
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune,4500
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai,50000
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune,55000
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore,55000
1009,2026-01-13,C108,Mouse,Accessories,2,800,Unknown,1600
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai,12000
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi,1600


In [0]:
print("Silver record count:", silver_df.count())

print("Duplicate order count:",
      silver_df.count() -
      silver_df.select("Order_ID").distinct().count())

silver_df.printSchema()

Silver record count: 7
Duplicate order count: 0
root
 |-- Order_ID: integer (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: integer (nullable = true)
 |-- City: string (nullable = false)
 |-- Revenue: integer (nullable = true)



In [0]:
silver_df = spark.table("workspace.default.silver_retail_sales")

display(silver_df)

Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City,Revenue
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune,4500
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai,50000
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune,55000
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore,55000
1009,2026-01-13,C108,Mouse,Accessories,2,800,Unknown,1600
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai,12000
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi,1600


In [0]:
from pyspark.sql.functions import sum, count, desc

product_sales = silver_df.groupBy("Product") \
    .agg(
        sum("Revenue").alias("Total_Revenue"),
        sum("Quantity").alias("Total_Quantity"),
        count("Order_ID").alias("Total_Orders")
    ) \
    .orderBy(desc("Total_Revenue"))

display(product_sales)

Product,Total_Revenue,Total_Quantity,Total_Orders
Laptop,110000,2,2
Mobile,50000,2,1
Monitor,12000,1,1
Keyboard,4500,3,1
Mouse,3200,4,2
